In [ ]:
########################################################################
# Inclusão das Bibliotecas Necessárias
########################################################################
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
########################################################################
# Localizando o Diretório Base
########################################################################
%cd /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código

In [ ]:
"""
VERSÃO VDN COM FALHAS ALEATÓRIAS DOS ROBÔS
- Algoritmo: Value Decomposition Networks (VDN)
  Q_tot = Q_1 + Q_2  (soma simples, sem mixer)
- Mesmos hiperparâmetros e ambiente do experimento QMIX para comparação direta
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from collections import deque
import random
from typing import List, Tuple, Dict, Optional
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from datetime import datetime
import json
import os
from pathlib import Path
import warnings
import time
import gc

import matplotlib
matplotlib.use('Agg')

warnings.filterwarnings('ignore')

# ==================== CONFIGURAÇÃO ====================
MAP_CONFIG = {
    'height': 12,
    'width': 8,
    'grid': [
        ['R1', '0', '0', '0', '0', '0', '0', 'R2'],
        ['0', '0', 'A', 'A', 'A', 'A', '0', '0'],
        ['0', '0', 'A', 'A', 'A', 'A', '0', '0'],
        ['X', '0', '0', '0', 'X', '0', '0', '0'],
        ['0', '0', 'X', '0', '0', '0', '0', '0'],
        ['0', '0', '0', 'X', '0', '0', 'X', 'X'],
        ['0', 'X', '0', '0', '0', 'X', '0', '0'],
        ['0', '0', '0', 'X', '0', '0', '0', '0'],
        ['X', '0', '0', '0', '0', '0', 'X', '0'],
        ['X', '0', 'B', 'B', 'B', 'B', 'X', '0'],
        ['X', '0', 'B', 'B', 'B', 'B', '0', '0'],
        ['0', '0', '0', '0', '0', '0', '0', '0']
    ]
}

class Config:
    # Exploração
    EPSILON_START = 1.0
    EPSILON_END = 0.05
    EPSILON_DECAY_STEPS = 250000

    # Aprendizado
    LEARNING_RATE = 0.0001
    BATCH_SIZE = 256
    GAMMA = 0.995
    TAU = 0.005
    WEIGHT_DECAY = 1e-5

    # Rede Neural
    HIDDEN_DIM = 512
    DROPOUT_RATE = 0.0

    # Memória
    BUFFER_SIZE = 200000
    PRIORITIZED_REPLAY = True
    ALPHA = 0.6
    BETA = 0.4

    # Treinamento
    MAX_GRAD_NORM = 1.0
    LEARNING_STARTS = 1000
    TRAIN_FREQ = 4
    USE_SOFT_UPDATE = True
    MAX_STEPS = 500
    EPISODES_PER_SESSION = 5000

    # Sistema
    SAVE_INTERVAL = 100
    CLEAN_MEMORY_EVERY = 500

    # Configurações de falha
    FAILURE_PROBABILITY = 0.05
    FAILURE_PENALTY = -0.10
    ALTERNATIVE_DIRECTIONS = True
    LOG_FAILURES = False


# ==================== PRIORITIZED REPLAY BUFFER PARA VDN ====================
# VDN não precisa de estado global para o mixing, apenas obs individuais
class VDNPrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6, beta=0.4):
        self.capacity = capacity
        self.alpha = alpha
        self.beta = beta
        self.buffer = []
        self.priorities = []
        self.position = 0

    def push(self, state, actions, rewards, next_state, done):
        max_priority = max(self.priorities) if self.priorities else 1.0

        if len(self.buffer) < self.capacity:
            self.buffer.append((state, actions, rewards, next_state, done))
            self.priorities.append(max_priority)
        else:
            self.buffer[self.position] = (state, actions, rewards, next_state, done)
            self.priorities[self.position] = max_priority

        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        if len(self.buffer) == self.capacity:
            priorities = np.array(self.priorities)
        else:
            priorities = np.array(self.priorities[:len(self.buffer)])

        probs = priorities ** self.alpha
        probs /= probs.sum()

        indices = np.random.choice(len(self.buffer), batch_size, p=probs)

        total = len(self.buffer)
        weights = (total * probs[indices]) ** (-self.beta)
        weights /= weights.max()

        batch = [self.buffer[idx] for idx in indices]
        states, actions, rewards, next_states, dones = zip(*batch)

        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones), indices, weights)

    def update_priorities(self, indices, td_errors):
        for idx, td_error in zip(indices, td_errors):
            self.priorities[idx] = abs(td_error) + 1e-6

    def __len__(self):
        return len(self.buffer)


# ==================== REDE NEURAL (DQN individual por agente) ====================
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=256, dropout=0.2):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.constant_(module.bias, 0.0)

    def forward(self, x):
        return self.network(x)


# ==================== AGENTE VDN ====================
class VDNAgent:
    def __init__(self, agent_id, state_dim, action_dim, config):
        self.agent_id = agent_id
        self.action_dim = action_dim
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.failure_count = 0
        self.successful_moves = 0

        self.policy_net = DQN(state_dim, action_dim, config.HIDDEN_DIM, config.DROPOUT_RATE).to(self.device)
        self.target_net = DQN(state_dim, action_dim, config.HIDDEN_DIM, config.DROPOUT_RATE).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())

        self.optimizer = optim.Adam(self.policy_net.parameters(),
                                   lr=config.LEARNING_RATE,
                                   weight_decay=config.WEIGHT_DECAY)

        self.steps_done = 0
        self.learning_steps = 0
        self.total_episodes = 0
        self.losses = []

    def get_epsilon(self):
        if self.steps_done >= self.config.EPSILON_DECAY_STEPS:
            return self.config.EPSILON_END

        epsilon = self.config.EPSILON_START - (self.config.EPSILON_START - self.config.EPSILON_END) * \
                  self.steps_done / self.config.EPSILON_DECAY_STEPS
        return max(self.config.EPSILON_END, epsilon)

    def select_action(self, state, training=True):
        self.steps_done += 1

        if training and random.random() < self.get_epsilon():
            return random.randrange(self.action_dim)

        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            q_values = self.policy_net(state_tensor)
            return q_values.argmax().item()

    def get_q_values(self, states):
        if isinstance(states, np.ndarray):
            states_tensor = torch.FloatTensor(states).to(self.device)
        else:
            states_tensor = states.to(self.device)
        return self.policy_net(states_tensor)

    def get_target_q_values(self, states):
        if isinstance(states, np.ndarray):
            states_tensor = torch.FloatTensor(states).to(self.device)
        else:
            states_tensor = states.to(self.device)
        return self.target_net(states_tensor)

    def soft_update_target(self):
        for target_param, policy_param in zip(self.target_net.parameters(),
                                               self.policy_net.parameters()):
            target_param.data.copy_(self.config.TAU * policy_param.data +
                                   (1 - self.config.TAU) * target_param.data)

    def reset_failure_stats(self):
        self.failure_count = 0
        self.successful_moves = 0


# ==================== AMBIENTE WAREHOUSE ====================
class WarehouseEnv(gym.Env):
    metadata = {'render.modes': ['rgb_array']}

    def __init__(self, config=None):
        super().__init__()

        self.config = config or Config()
        self.height = MAP_CONFIG['height']
        self.width = MAP_CONFIG['width']
        self.grid = [row[:] for row in MAP_CONFIG['grid']]

        self.robot_positions = None
        self.box_positions = None
        self.targets = self._find_positions('B')

        self.num_robots = 2
        self.num_boxes = len(self._find_positions('A'))
        self.num_targets = len(self.targets)

        self.delivered_boxes = None
        self.steps = 0
        self.max_steps = self.config.MAX_STEPS
        self.total_deliveries = 0
        self.collisions = 0
        self.failures = [0, 0]
        self.distance_traveled = [0, 0]
        self.previous_distances = None

        self.movement_log = []

        self.action_space = spaces.Tuple([spaces.Discrete(6) for _ in range(self.num_robots)])

        obs_dim = (self.num_robots * 2) + (self.num_boxes * 2) + (self.num_targets * 2) + 8
        self.observation_space = spaces.Box(
            low=-1, high=self.height + self.width,
            shape=(obs_dim,),
            dtype=np.float32
        )

        self.frame_buffer = []

    def _find_positions(self, symbols):
        if isinstance(symbols, str):
            symbols = [symbols]
        positions = []
        for i in range(self.height):
            for j in range(self.width):
                cell = self.grid[i][j]
                if any(cell.startswith(sym) for sym in symbols):
                    positions.append((i, j))
        return positions

    def reset(self):
        self.grid = [row[:] for row in MAP_CONFIG['grid']]
        self.robot_positions = self._find_positions('R')
        self.box_positions = self._find_positions('A')
        self.delivered_boxes = [False] * self.num_boxes

        self.steps = 0
        self.total_deliveries = 0
        self.collisions = 0
        self.failures = [0, 0]
        self.distance_traveled = [0, 0]
        self.movement_log = []

        self.previous_distances = [self._min_distance_to_boxes(r) for r in range(self.num_robots)]

        return self._get_observation(), self._get_info()

    def _min_distance_to_boxes(self, robot_id):
        robot_pos = self.robot_positions[robot_id]
        remaining_boxes = [box_pos for box_pos in self.box_positions
                          if box_pos is not None and
                          not self.delivered_boxes[self.box_positions.index(box_pos)]]

        if not remaining_boxes:
            return 0
        return min([abs(robot_pos[0] - box_pos[0]) + abs(robot_pos[1] - box_pos[1])
                   for box_pos in remaining_boxes])

    def _is_valid_position(self, pos, robot_id=None):
        i, j = pos
        if i < 0 or i >= self.height or j < 0 or j >= self.width:
            return False

        cell = self.grid[i][j]
        if cell == 'X':
            return False

        if robot_id is not None:
            for rid, rpos in enumerate(self.robot_positions):
                if rid != robot_id and rpos == (i, j):
                    return False
        return True

    def _get_direction_name(self, action):
        direcoes = {0: "CIMA", 1: "BAIXO", 2: "ESQUERDA", 3: "DIREITA", 4: "PEGAR", 5: "SOLTAR", 6: "PARADO"}
        return direcoes.get(action, "DESCONHECIDO")

    def _get_alternative_direction(self, original_action, robot_id):
        i, j = self.robot_positions[robot_id]
        alternative_actions = [a for a in range(4) if a != original_action]
        random.shuffle(alternative_actions)

        for alt_action in alternative_actions:
            alt_i, alt_j = i, j
            if alt_action == 0: alt_i -= 1
            elif alt_action == 1: alt_i += 1
            elif alt_action == 2: alt_j -= 1
            elif alt_action == 3: alt_j += 1

            if self._is_valid_position((alt_i, alt_j), robot_id):
                return alt_action, (alt_i, alt_j)

        return None, None

    def _move_robot_with_failure(self, robot_id, action):
        i, j = self.robot_positions[robot_id]
        new_i, new_j = i, j
        movimento_original = self._get_direction_name(action)

        if action == 0: new_i -= 1
        elif action == 1: new_i += 1
        elif action == 2: new_j -= 1
        elif action == 3: new_j += 1

        desired_valid = self._is_valid_position((new_i, new_j), robot_id)
        movimento_final = movimento_original
        pos_final = (new_i, new_j)
        falha_ocorreu = False

        if random.random() < self.config.FAILURE_PROBABILITY:
            self.failures[robot_id] += 1
            falha_ocorreu = True

            if self.config.ALTERNATIVE_DIRECTIONS:
                alt_action, alt_pos = self._get_alternative_direction(action, robot_id)
                if alt_action is not None:
                    movimento_final = self._get_direction_name(alt_action)
                    pos_final = alt_pos

                    old_pos = self.robot_positions[robot_id]
                    self.grid[old_pos[0]][old_pos[1]] = '0'
                    self.grid[alt_pos[0]][alt_pos[1]] = f'R{robot_id + 1}'
                    self.robot_positions[robot_id] = alt_pos
                    self.distance_traveled[robot_id] += 1

                    self.movement_log.append({
                        'robo': f'R{robot_id+1}',
                        'pos_original_x': i,
                        'pos_original_y': j,
                        'pos_final_x': alt_pos[0],
                        'pos_final_y': alt_pos[1],
                        'movimento_original': movimento_original,
                        'movimento_final': movimento_final,
                        'falha': falha_ocorreu
                    })

                    return self.config.FAILURE_PENALTY - 0.01

            self.movement_log.append({
                'robo': f'R{robot_id+1}',
                'pos_original_x': i,
                'pos_original_y': j,
                'pos_final_x': i,
                'pos_final_y': j,
                'movimento_original': movimento_original,
                'movimento_final': "PARADO",
                'falha': falha_ocorreu
            })
            return self.config.FAILURE_PENALTY

        if desired_valid:
            self.distance_traveled[robot_id] += 1
            old_pos = self.robot_positions[robot_id]
            self.grid[old_pos[0]][old_pos[1]] = '0'
            self.grid[new_i][new_j] = f'R{robot_id + 1}'
            self.robot_positions[robot_id] = (new_i, new_j)

            self.movement_log.append({
                'robo': f'R{robot_id+1}',
                'pos_original_x': i,
                'pos_original_y': j,
                'pos_final_x': new_i,
                'pos_final_y': new_j,
                'movimento_original': movimento_original,
                'movimento_final': movimento_original,
                'falha': falha_ocorreu
            })
            return -0.005
        else:
            self.collisions += 1
            self.movement_log.append({
                'robo': f'R{robot_id+1}',
                'pos_original_x': i,
                'pos_original_y': j,
                'pos_final_x': i,
                'pos_final_y': j,
                'movimento_original': movimento_original,
                'movimento_final': "BLOQUEADO",
                'falha': falha_ocorreu
            })
            return -0.02

    def _pickup_box(self, robot_id):
        robot_pos = self.robot_positions[robot_id]
        for box_id, box_pos in enumerate(self.box_positions):
            if not self.delivered_boxes[box_id] and box_pos == robot_pos:
                self.box_positions[box_id] = None
                self.grid[robot_pos[0]][robot_pos[1]] = f'R{robot_id + 1}'

                self.movement_log.append({
                    'robo': f'R{robot_id+1}',
                    'pos_original_x': robot_pos[0],
                    'pos_original_y': robot_pos[1],
                    'pos_final_x': robot_pos[0],
                    'pos_final_y': robot_pos[1],
                    'movimento_original': "PEGAR",
                    'movimento_final': "PEGOU",
                    'falha': False
                })
                return 2.0
        return -0.02

    def _drop_box(self, robot_id):
        robot_pos = self.robot_positions[robot_id]

        box_with_robot = None
        for box_id, box_pos in enumerate(self.box_positions):
            if box_pos is None and not self.delivered_boxes[box_id]:
                box_with_robot = box_id
                break

        if box_with_robot is None:
            return -0.02

        for target_pos in self.targets:
            if robot_pos == target_pos:
                self.delivered_boxes[box_with_robot] = True
                self.total_deliveries += 1
                self.grid[robot_pos[0]][robot_pos[1]] = f'R{robot_id + 1}'

                self.movement_log.append({
                    'robo': f'R{robot_id+1}',
                    'pos_original_x': robot_pos[0],
                    'pos_original_y': robot_pos[1],
                    'pos_final_x': robot_pos[0],
                    'pos_final_y': robot_pos[1],
                    'movimento_original': "SOLTAR",
                    'movimento_final': "ENTREGOU",
                    'falha': False
                })
                return 25.0

        return -0.05

    def _calculate_shaped_reward(self, robot_id, base_reward):
        reward = base_reward

        current_distance = self._min_distance_to_boxes(robot_id)
        previous_distance = self.previous_distances[robot_id]

        if current_distance < previous_distance:
            reward += 0.1 * (previous_distance - current_distance)
        elif current_distance > previous_distance:
            reward -= 0.02 * (current_distance - previous_distance)

        self.previous_distances[robot_id] = current_distance

        if all(self.delivered_boxes):
            reward += 50.0

        return reward

    def _get_observation(self):
        obs = []

        for robot_pos in self.robot_positions:
            obs.append(robot_pos[0] / self.height)
            obs.append(robot_pos[1] / self.width)

        for box_id, box_pos in enumerate(self.box_positions):
            if box_pos is None or self.delivered_boxes[box_id]:
                obs.append(-1)
                obs.append(-1)
            else:
                obs.append(box_pos[0] / self.height)
                obs.append(box_pos[1] / self.width)

        for target_pos in self.targets:
            obs.append(target_pos[0] / self.height)
            obs.append(target_pos[1] / self.width)

        for robot_pos in self.robot_positions:
            min_box_dist = min([abs(robot_pos[0] - box_pos[0]) + abs(robot_pos[1] - box_pos[1])
                               for box_pos in self.box_positions
                               if box_pos is not None and
                               not self.delivered_boxes[self.box_positions.index(box_pos)]],
                              default=100)
            obs.append(min_box_dist / (self.height + self.width))

            min_target_dist = min([abs(robot_pos[0] - target_pos[0]) + abs(robot_pos[1] - target_pos[1])
                                  for target_pos in self.targets],
                                 default=100)
            obs.append(min_target_dist / (self.height + self.width))

        return np.array(obs, dtype=np.float32)

    def step(self, actions):
        self.steps += 1

        if len(actions) != self.num_robots:
            actions = [actions] * self.num_robots

        total_reward = 0
        rewards = [0, 0]

        movement_rewards = []
        for robot_id, action in enumerate(actions):
            if action < 4:
                reward = self._move_robot_with_failure(robot_id, action)
                movement_rewards.append(reward)
            else:
                movement_rewards.append(0)

        interaction_rewards = []
        for robot_id, action in enumerate(actions):
            if action == 4:
                reward = self._pickup_box(robot_id)
                interaction_rewards.append(reward)
            elif action == 5:
                reward = self._drop_box(robot_id)
                interaction_rewards.append(reward)
            else:
                interaction_rewards.append(0)

        for robot_id in range(self.num_robots):
            base_reward = movement_rewards[robot_id] + interaction_rewards[robot_id]
            shaped_reward = self._calculate_shaped_reward(robot_id, base_reward)
            rewards[robot_id] = shaped_reward
            total_reward += shaped_reward

        terminated = all(self.delivered_boxes)
        truncated = self.steps >= self.max_steps

        observation = self._get_observation()
        info = self._get_info()

        return observation, rewards, terminated, truncated, info

    def _get_info(self):
        return {
            'steps': self.steps,
            'total_deliveries': self.total_deliveries,
            'collisions': self.collisions,
            'failures_r1': self.failures[0],
            'failures_r2': self.failures[1],
            'distance_traveled': self.distance_traveled.copy(),
            'remaining_boxes': sum(1 for d in self.delivered_boxes if not d),
            'success_rate': self.total_deliveries / self.num_boxes if self.steps > 0 else 0,
        }

    def get_movement_log(self):
        return self.movement_log

    def close(self):
        pass


# ==================== VDN TRAINER ====================
# A diferença central do VDN: Q_tot = Q_1 + Q_2 (soma simples)
# Sem QMixer, sem hypernetwork, sem estado global no mixing
class VDNTrainer:
    def __init__(self, agents, config, state_dim):
        self.agents = agents
        self.config = config
        self.state_dim = state_dim
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.memory = VDNPrioritizedReplayBuffer(config.BUFFER_SIZE, alpha=config.ALPHA)
        self.learning_steps = 0

    def remember(self, state, actions, rewards, next_state, done):
        self.memory.push(state, actions, rewards, next_state, done)

    def optimize(self):
        if len(self.memory) < self.config.BATCH_SIZE:
            return 0

        self.learning_steps += 1

        if self.learning_steps % self.config.TRAIN_FREQ != 0:
            return 0

        (states, actions, rewards, next_states, dones, indices, weights) = self.memory.sample(self.config.BATCH_SIZE)

        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)
        weights = torch.FloatTensor(weights).to(self.device)

        # Zerar gradientes de todos os agentes antes do forward pass
        for agent in self.agents:
            agent.optimizer.zero_grad()

        # Q-values correntes: Q_i(s_i, a_i) para cada agente
        curr_qs = []
        for i, agent in enumerate(self.agents):
            agent_qs = agent.get_q_values(states)
            curr_qs.append(agent_qs.gather(1, actions[:, i].unsqueeze(1)).squeeze(1))

        # VDN: Q_tot = soma simples dos Q-values individuais
        curr_q_total = torch.stack(curr_qs, dim=1).sum(dim=1, keepdim=True)

        # Q-values alvo: max_a Q_i_target(s_i', a) para cada agente
        with torch.no_grad():
            target_qs = []
            for agent in self.agents:
                agent_target_qs = agent.get_target_q_values(next_states)
                target_qs.append(agent_target_qs.max(1)[0])

            # VDN: target_Q_tot = soma dos targets individuais
            target_q_total = torch.stack(target_qs, dim=1).sum(dim=1, keepdim=True)
            target = rewards.sum(dim=1, keepdim=True) + self.config.GAMMA * target_q_total * (1 - dones.unsqueeze(1))

        td_errors = (target - curr_q_total).squeeze()
        loss = (weights * td_errors.pow(2)).mean()

        # Backprop único flui pelas redes de todos os agentes
        loss.backward()

        for agent in self.agents:
            torch.nn.utils.clip_grad_norm_(agent.policy_net.parameters(), self.config.MAX_GRAD_NORM)
            agent.optimizer.step()
            agent.soft_update_target()

        priorities = td_errors.abs().detach().cpu().numpy() + 1e-6
        self.memory.update_priorities(indices, priorities)

        return loss.item()


# ==================== FUNÇÕES DE PLOTAGEM ====================
def plot_individual_graphs(metrics, save_dir):
    """Plota e salva gráficos individuais"""

    print(f"\n📊 Gerando gráficos em: {save_dir}")

    episodes = range(1, len(metrics['episode_rewards']) + 1)

    if len(episodes) == 0:
        print("⚠️ Nenhum dado para plotar!")
        return

    # 1. Recompensa por episódio
    try:
        plt.figure(figsize=(12, 6))
        plt.plot(episodes, metrics['episode_rewards'], 'b-', alpha=0.5, linewidth=1)
        if len(metrics['episode_rewards']) >= 100:
            moving_avg = np.convolve(metrics['episode_rewards'], np.ones(100)/100, mode='valid')
            plt.plot(range(100, len(metrics['episode_rewards']) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel (100)')
        plt.xlabel('Episódio')
        plt.ylabel('Recompensa')
        plt.title('Recompensa por Episódio (VDN)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(save_dir / 'grafico_recompensa.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✅ grafico_recompensa.png salvo")
    except Exception as e:
        print(f"  ❌ Erro ao salvar grafico_recompensa.png: {e}")

    # 2. Entregas por episódio
    try:
        plt.figure(figsize=(12, 6))
        plt.plot(episodes, metrics['episode_deliveries'], 'g-', alpha=0.5, linewidth=1)
        if len(metrics['episode_deliveries']) >= 100:
            moving_avg = np.convolve(metrics['episode_deliveries'], np.ones(100)/100, mode='valid')
            plt.plot(range(100, len(metrics['episode_deliveries']) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel (100)')
        plt.axhline(y=8, color='orange', linestyle='--', label='Meta (8 caixas)')
        plt.xlabel('Episódio')
        plt.ylabel('Entregas')
        plt.title('Entregas por Episódio (VDN)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(save_dir / 'grafico_entregas.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✅ grafico_entregas.png salvo")
    except Exception as e:
        print(f"  ❌ Erro ao salvar grafico_entregas.png: {e}")

    # 3. Steps por episódio
    try:
        plt.figure(figsize=(12, 6))
        plt.plot(episodes, metrics['episode_steps'], 'orange', alpha=0.5, linewidth=1)
        if len(metrics['episode_steps']) >= 100:
            moving_avg = np.convolve(metrics['episode_steps'], np.ones(100)/100, mode='valid')
            plt.plot(range(100, len(metrics['episode_steps']) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel (100)')
        plt.xlabel('Episódio')
        plt.ylabel('Steps')
        plt.title('Steps por Episódio (VDN)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(save_dir / 'grafico_steps.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✅ grafico_steps.png salvo")
    except Exception as e:
        print(f"  ❌ Erro ao salvar grafico_steps.png: {e}")

    # 4. Taxa de Sucesso por episódio
    try:
        plt.figure(figsize=(12, 6))
        plt.plot(episodes, metrics['success_rates'], 'purple', alpha=0.5, linewidth=1)
        if len(metrics['success_rates']) >= 100:
            moving_avg = np.convolve(metrics['success_rates'], np.ones(100)/100, mode='valid')
            plt.plot(range(100, len(metrics['success_rates']) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel (100)')
        plt.axhline(y=0.95, color='green', linestyle='--', label='Meta 95%')
        plt.xlabel('Episódio')
        plt.ylabel('Taxa de Sucesso')
        plt.title('Taxa de Sucesso por Episódio (VDN)')
        plt.ylim([0, 1])
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(save_dir / 'grafico_taxa_sucesso.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✅ grafico_taxa_sucesso.png salvo")
    except Exception as e:
        print(f"  ❌ Erro ao salvar grafico_taxa_sucesso.png: {e}")

    # 5. Colisões por episódio
    try:
        plt.figure(figsize=(12, 6))
        plt.plot(episodes, metrics['collisions'], 'red', alpha=0.5, linewidth=1)
        if len(metrics['collisions']) >= 100:
            moving_avg = np.convolve(metrics['collisions'], np.ones(100)/100, mode='valid')
            plt.plot(range(100, len(metrics['collisions']) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel (100)')
        plt.xlabel('Episódio')
        plt.ylabel('Colisões')
        plt.title('Colisões por Episódio (VDN)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(save_dir / 'grafico_colisoes.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✅ grafico_colisoes.png salvo")
    except Exception as e:
        print(f"  ❌ Erro ao salvar grafico_colisoes.png: {e}")

    # 6. Falhas por episódio
    try:
        plt.figure(figsize=(12, 6))
        failures_total = np.array(metrics['failures_r1']) + np.array(metrics['failures_r2'])
        plt.plot(episodes, failures_total, 'brown', alpha=0.5, linewidth=1, label='Total')
        plt.plot(episodes, metrics['failures_r1'], 'red', alpha=0.5, linewidth=1, label='R1')
        plt.plot(episodes, metrics['failures_r2'], 'blue', alpha=0.5, linewidth=1, label='R2')
        if len(failures_total) >= 100:
            moving_avg = np.convolve(failures_total, np.ones(100)/100, mode='valid')
            plt.plot(range(100, len(failures_total) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel Total')
        plt.xlabel('Episódio')
        plt.ylabel('Falhas')
        plt.title('Falhas por Episódio (VDN)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(save_dir / 'grafico_falhas.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✅ grafico_falhas.png salvo")
    except Exception as e:
        print(f"  ❌ Erro ao salvar grafico_falhas.png: {e}")

    print(f"📊 Todos os gráficos foram salvos em: {save_dir}")


def save_metrics_csv(metrics, save_dir):
    """Salva as métricas em arquivo CSV"""
    df = pd.DataFrame({
        'episodio': range(1, len(metrics['episode_rewards']) + 1),
        'recompensa': metrics['episode_rewards'],
        'entregas': metrics['episode_deliveries'],
        'steps': metrics['episode_steps'],
        'taxa_sucesso': metrics['success_rates'],
        'colisoes': metrics['collisions'],
        'falha_r1': metrics['failures_r1'],
        'falha_r2': metrics['failures_r2']
    })
    df.to_csv(save_dir / 'metricas_treinamento.csv', index=False)
    print(f"📊 Métricas salvas em: {save_dir / 'metricas_treinamento.csv'}")


def save_movement_log(movement_log, save_dir):
    """Salva o log de movimentos em arquivo CSV"""
    if movement_log:
        df = pd.DataFrame(movement_log)
        df.to_csv(save_dir / 'log_movimentos.csv', index=False)
        print(f"📊 Log de movimentos salvo em: {save_dir / 'log_movimentos.csv'}")
    else:
        print(f"⚠️ Nenhum movimento registrado")


# ==================== FUNÇÃO DE TREINAMENTO ====================
def train_single_session(session_id, num_episodes, config, agents, trainer, save_dir):
    env = WarehouseEnv(config=config)

    metrics = {
        'episode_rewards': [],
        'episode_deliveries': [],
        'episode_steps': [],
        'success_rates': [],
        'collisions': [],
        'failures_r1': [],
        'failures_r2': [],
        'vdn_losses': [],
    }

    all_movement_logs = []

    best_reward = -float('inf')
    session_start_time = time.time()

    print(f"\n{'='*60}")
    print(f"🎯 SESSÃO VDN COM FALHAS {session_id} - {num_episodes} episódios")
    print(f"   ⚠️ Probabilidade de falha: {config.FAILURE_PROBABILITY*100:.0f}%")
    print(f"{'='*60}")

    for episode in range(num_episodes):
        obs, _ = env.reset()
        episode_reward = 0
        episode_collisions = 0

        for agent in agents:
            agent.reset_failure_stats()

        for step in range(config.MAX_STEPS):
            actions = [agent.select_action(obs) for agent in agents]
            next_obs, rewards, terminated, truncated, info = env.step(actions)

            # VDN: sem estado global no buffer
            trainer.remember(obs, actions, rewards, next_obs, terminated or truncated)

            episode_reward += sum(rewards)
            episode_collisions = info['collisions']

            vdn_loss = trainer.optimize()
            if vdn_loss and vdn_loss > 0:
                metrics['vdn_losses'].append(vdn_loss)

            obs = next_obs

            if terminated or truncated:
                break

        metrics['episode_rewards'].append(episode_reward)
        metrics['episode_deliveries'].append(info['total_deliveries'])
        metrics['episode_steps'].append(step + 1)
        metrics['success_rates'].append(info['success_rate'])
        metrics['collisions'].append(episode_collisions)
        metrics['failures_r1'].append(info['failures_r1'])
        metrics['failures_r2'].append(info['failures_r2'])

        all_movement_logs.extend(env.get_movement_log())

        if (episode + 1) % 100 == 0:
            recent_rewards = metrics['episode_rewards'][-100:]
            recent_deliveries = metrics['episode_deliveries'][-100:]
            epsilon = agents[0].get_epsilon()
            elapsed = time.time() - session_start_time

            print(f"  Sessão {session_id} | Ep {episode+1:4d}/{num_episodes} | "
                  f"Reward: {np.mean(recent_rewards):7.2f} | "
                  f"Entregas: {np.mean(recent_deliveries):.2f} | "
                  f"ε: {epsilon:.3f} | "
                  f"Tempo: {elapsed/60:.1f}min")

        if (episode + 1) % config.CLEAN_MEMORY_EVERY == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    env.close()

    save_movement_log(all_movement_logs, save_dir)

    session_stats = {
        'session_id': session_id,
        'total_episodes': num_episodes,
        'best_reward': max(metrics['episode_rewards']),
        'avg_reward_last_100': np.mean(metrics['episode_rewards'][-100:]),
        'avg_deliveries_last_100': np.mean(metrics['episode_deliveries'][-100:]),
        'final_success_rate': metrics['success_rates'][-1],
        'total_failures': sum(metrics['failures_r1']) + sum(metrics['failures_r2']),
        'total_time_minutes': (time.time() - session_start_time) / 60
    }

    return metrics, session_stats


# ==================== FUNÇÃO PRINCIPAL ====================
def run_multi_session_training(num_sessions=3, episodes_per_session=3000):
    config = Config()
    config.EPISODES_PER_SESSION = episodes_per_session

    print("=" * 80)
    print("🏭 TREINAMENTO VDN COM FALHAS - WAREHOUSE")
    print("=" * 80)
    print(f"\n📋 CONFIGURAÇÃO:")
    print(f"   • Algoritmo: VDN (Value Decomposition Networks) com Falhas")
    print(f"   • Mixing: Q_tot = Q_1 + Q_2 (soma simples, sem mixer)")
    print(f"   • Dispositivo: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
    print(f"   • Sessões: {num_sessions}")
    print(f"   • Episódios por sessão: {episodes_per_session}")
    print(f"   • Total de episódios: {num_sessions * episodes_per_session}")
    print(f"   • ⚠️ PROBABILIDADE DE FALHA: {config.FAILURE_PROBABILITY*100:.0f}%")
    print("=" * 80)

    temp_env = WarehouseEnv(config=config)
    sample_obs, _ = temp_env.reset()
    state_dim = len(sample_obs)
    action_dim = temp_env.action_space[0].n
    temp_env.close()

    print(f"\n📊 Dimensões:")
    print(f"   • Estado individual: {state_dim}")
    print(f"   • Ação: {action_dim}")
    print(f"   • Robôs: 2")
    print(f"   • Caixas: 8")
    print(f"   • Mixer: Nenhum (soma simples)")

    agents = [VDNAgent(i, state_dim, action_dim, config) for i in range(2)]
    trainer = VDNTrainer(agents, config, state_dim)

    all_metrics = []
    all_session_stats = []
    consolidated_metrics = {
        'episode_rewards': [],
        'episode_deliveries': [],
        'episode_steps': [],
        'success_rates': [],
        'collisions': [],
        'failures_r1': [],
        'failures_r2': []
    }

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_dir = Path(f"vdn_failure_results_{timestamp}")
    results_dir.mkdir(exist_ok=True)
    print(f"\n📁 Diretório de resultados criado: {results_dir.absolute()}")

    total_start_time = time.time()

    for session_id in range(1, num_sessions + 1):
        session_metrics, session_stats = train_single_session(
            session_id, episodes_per_session, config, agents, trainer, results_dir
        )

        all_metrics.append(session_metrics)
        all_session_stats.append(session_stats)

        for key in consolidated_metrics:
            consolidated_metrics[key].extend(session_metrics[key])

        print(f"\n📊 RESUMO SESSÃO {session_id}:")
        print(f"   • Melhor reward: {session_stats['best_reward']:.2f}")
        print(f"   • Média entregas (últimos 100): {session_stats['avg_deliveries_last_100']:.2f}")
        print(f"   • Taxa sucesso final: {session_stats['final_success_rate']:.1%}")
        print(f"   • Total falhas: {session_stats['total_failures']}")
        print(f"   • Tempo: {session_stats['total_time_minutes']:.1f} min")

    total_time = (time.time() - total_start_time) / 60

    print(f"\n💾 SALVANDO RESULTADOS...")

    save_metrics_csv(consolidated_metrics, results_dir)

    print(f"\n📊 Gerando gráficos...")
    plot_individual_graphs(consolidated_metrics, results_dir)

    df_stats = pd.DataFrame(all_session_stats)
    df_stats.to_csv(results_dir / "session_stats.csv", index=False)

    models_dir = results_dir / "models"
    models_dir.mkdir(exist_ok=True)
    for i, agent in enumerate(agents):
        torch.save(agent.policy_net.state_dict(), models_dir / f"vdn_agent_{i}_final.pth")

    report = f"""
    ========================================
    RELATÓRIO FINAL DO TREINAMENTO VDN
    ========================================

    DATA: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    DIRETÓRIO: {results_dir.absolute()}

    CONFIGURAÇÃO:
    - Algoritmo: VDN (Value Decomposition Networks)
    - Mixing: Q_tot = Q_1 + Q_2 (soma simples)
    - Total de sessões: {len(all_session_stats)}
    - Total de episódios: {len(consolidated_metrics['episode_rewards'])}
    - Tempo total: {total_time:.1f} minutos
    - Probabilidade de falha: {config.FAILURE_PROBABILITY*100:.0f}%

    MÉTRICAS GERAIS:
    - Melhor recompensa: {max(consolidated_metrics['episode_rewards']):.2f}
    - Média recompensa (últimos 100): {np.mean(consolidated_metrics['episode_rewards'][-100:]):.2f}

    ENTREGAS:
    - Média entregas (últimos 100): {np.mean(consolidated_metrics['episode_deliveries'][-100:]):.2f}/8
    - Taxa de sucesso final: {consolidated_metrics['success_rates'][-1]:.1%}

    FALHAS:
    - Total de falhas: {sum(consolidated_metrics['failures_r1']) + sum(consolidated_metrics['failures_r2'])}
    - Média falhas por episódio: {np.mean(np.array(consolidated_metrics['failures_r1']) + np.array(consolidated_metrics['failures_r2'])):.2f}

    GRÁFICOS GERADOS:
    - grafico_recompensa.png
    - grafico_entregas.png
    - grafico_steps.png
    - grafico_taxa_sucesso.png
    - grafico_colisoes.png
    - grafico_falhas.png

    ========================================
    """

    with open(results_dir / "relatorio_final.txt", 'w', encoding='utf-8') as f:
        f.write(report)

    print(f"\n✅ TREINAMENTO VDN CONCLUÍDO!")
    print(f"📁 Resultados salvos em: {results_dir.absolute()}")
    print(f"⏱️ Tempo total: {total_time:.1f} minutos")
    print(f"\n📦 Arquivos gerados:")
    print(f"   • metricas_treinamento.csv - Métricas por episódio")
    print(f"   • log_movimentos.csv - Log de movimentos dos robôs")
    print(f"   • session_stats.csv - Estatísticas por sessão")
    print(f"   • relatorio_final.txt - Relatório completo")
    print(f"   • grafico_recompensa.png, grafico_entregas.png, ...")
    print(f"   • models/vdn_agent_0_final.pth, models/vdn_agent_1_final.pth")

    return agents, consolidated_metrics


# ==================== EXECUÇÃO ====================
if __name__ == "__main__":
    NUM_SESSIONS = 3
    EPISODES_PER_SESSION = 3000

    print("\n" + "=" * 80)
    print("🎮 INICIANDO TREINAMENTO VDN COM FALHAS")
    print("=" * 80)

    try:
        agents, metrics = run_multi_session_training(
            num_sessions=NUM_SESSIONS,
            episodes_per_session=EPISODES_PER_SESSION
        )

        print("\n✨ TREINAMENTO VDN CONCLUÍDO COM SUCESSO! ✨")

    except Exception as e:
        print(f"\n❌ ERRO DURANTE O TREINAMENTO: {e}")
        import traceback
        traceback.print_exc()